# 第 2 章 统一不同的大模型

## 不止一个大模型

上一章，小明的学习助手通过 OpenAI 的 API 说出了第一句话。

没过几天，小明的同学跟他说：“你试过 Claude 吗？写代码特别好用。”小明一搜，发现能用的大模型还真不少：OpenAI 的 GPT、Anthropic 的 Claude、Google 的 Gemini，还有 DeepSeek、通义千问这些国产模型……

小明想：要是学习助手能随时换用不同的模型就好了。比如平时用便宜的小模型，遇到难题再换一个更强的。

问题是，每家厂商的 API 长得都不太一样。如果代码里到处都是“OpenAI 怎么调、Claude 怎么调”，以后每换一次模型，就要把代码翻个遍。

这一章，我们先学会调用 Claude，然后给不同厂商的 API 包上一层**统一的外壳**：不管用的是哪家的模型，调用方式都一模一样。

## 准备工作：Claude 的 API Key

调用 Claude 和调用 OpenAI 一样，需要一个开发者账号和 API Key，步骤也差不多：

1. 打开 Claude 的开发者平台 [platform.claude.com](https://platform.claude.com)，注册或登录。注意，这个开发者账号和聊天用的 claude.ai 是分开的。
2. 绑定付款方式，充值一小笔钱。和 OpenAI 一样，API 按 token 收费。
3. 进入 **Settings → API keys**，点击 **Create key**，起个名字，复制生成的 Key。Claude 的 Key 以 `sk-ant-` 开头，同样只显示一次。

然后像上一章一样，把 Key 保存到环境变量里，名字叫 `ANTHROPIC_API_KEY`（Anthropic 是开发 Claude 的公司）。

**macOS / Linux：**

```bash
echo 'export ANTHROPIC_API_KEY="sk-ant-..."' >> ~/.zshrc
source ~/.zshrc
```

**Windows（PowerShell）：**

```powershell
setx ANTHROPIC_API_KEY "sk-ant-..."
```

再安装 Anthropic 官方的 Python 库：

```bash
pip install anthropic
```

记得重新启动 Jupyter，然后检查一下两个 Key 是否都设置好了：

In [ ]:
import os

print("OPENAI_API_KEY" in os.environ, "ANTHROPIC_API_KEY" in os.environ)

## OpenAI 和 Claude 的 API 有什么不同

先把需要的模块都导入进来。`abc` 是 Python 自带的模块，马上就会用到；其余的分别是 OpenAI 和 Anthropic 的库。

In [ ]:
from abc import ABC, abstractmethod

import anthropic
from anthropic.types import UserMessageParam
from openai import OpenAI
from openai.types.chat import ChatCompletionUserMessageParam

两家的 API 大体相同：都要指定模型 `model`，都把对话放在 `messages` 列表里，每条消息都有 `role` 和 `content`。但细节上有几处不一样：

| | OpenAI | Claude |
|---|---|---|
| 调用方法 | `client.chat.completions.create(...)` | `client.messages.create(...)` |
| 最多生成多少 token | 可以不写 | **必须写** `max_tokens` |
| 回答在哪里 | `response.choices[0].message.content` | `response.content[0].text` |

这样的差别，后面还会遇到更多（比如第 3 章的系统提示词、第 9 章的工具调用）。如果让程序的其他部分直接面对这些差别，代码很快就会乱成一团。

## 用抽象类统一接口

解决办法是：先定一个“规矩”，规定所有的大模型都要提供同样的方法；再为每家厂商各写一个类，在类的内部处理它们各自的细节。

这个“规矩”，在 Python 里可以用**抽象类**来写。`LLMProvider` 继承了 `ABC`（Abstract Base Class，抽象基类），它只规定：每个大模型都必须有一个 `call` 方法，传入一句话 `message`，返回模型的回答。至于具体怎么调用，它不管。

`@abstractmethod` 表示这个方法必须由子类来实现。抽象类本身不能直接创建对象，如果某个子类忘了写 `call`，Python 也会报错提醒我们。

In [ ]:
class LLMProvider(ABC):

    @abstractmethod
    def call(self, message, model=None):
        pass

## OpenAI 的实现

`OpenAIProvider` 就是把上一章的代码搬进类里：

- `__init__` 创建客户端，并设置默认模型 `gpt-5.4-mini`。
- `call` 把传进来的一句话包装成消息，调用 API，再把回答的文字取出来返回。
- `model=None` 表示可以临时指定别的模型；不指定时，`model or self.default_model` 就会用默认模型。

In [ ]:
class OpenAIProvider(LLMProvider):

    def __init__(self):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.default_model = "gpt-5.4-mini"

    def call(self, message, model=None):
        response = self.client.chat.completions.create(
            model=model or self.default_model,
            messages=[ChatCompletionUserMessageParam(role="user", content=message)]
        )
        return response.choices[0].message.content

用起来非常简单，一行代码就能拿到回答：

In [ ]:
provider = OpenAIProvider()
print(provider.call("你好，我叫小明，正在自学编程。请用两三句话介绍一下你自己。"))

## Claude 的实现

`ClaudeProvider` 的结构和 `OpenAIProvider` 完全一样，只是在 `call` 内部换成了 Claude 的调用方式：

- 用 `anthropic.Anthropic` 创建客户端，Key 来自环境变量 `ANTHROPIC_API_KEY`。
- 默认模型是 `claude-haiku-4-5`，这是 Claude 家族里又快又便宜的模型。
- 调用时必须给出 `max_tokens`，这里默认最多生成 4096 个 token。
- 回答从 `response.content[0].text` 取出。

对外，它和 `OpenAIProvider` 一样，都只有一个 `call` 方法。

In [ ]:
class ClaudeProvider(LLMProvider):

    def __init__(self):
        self.client = anthropic.Anthropic(
            api_key=os.getenv("ANTHROPIC_API_KEY")
        )
        self.default_model = "claude-haiku-4-5"

    def call(self, message, model=None, max_tokens=4096):
        response = self.client.messages.create(
            model=model or self.default_model,
            max_tokens=max_tokens,
            messages=[UserMessageParam(role="user", content=message)]
        )
        return response.content[0].text

In [ ]:
provider = ClaudeProvider()
print(provider.call("你好，我叫小明，正在自学编程。请用两三句话介绍一下你自己。"))

## 随时切换模型

现在，两个模型的用法完全一样了。不管 `provider` 是哪一个，调用的代码都不用改：

In [ ]:
for provider in [OpenAIProvider(), ClaudeProvider()]:
    print(type(provider).__name__)
    print(provider.call("用一句话告诉我，学编程最重要的是什么？"))
    print()

这就是抽象类的好处：学习助手的其他代码只认识 `LLMProvider`，只管调用 `call`。以后想换模型，只需要换一个 `provider`；想支持新的厂商，只需要再写一个子类。从这一章开始，书里的每个例子都会同时提供 OpenAI 和 Claude 两个版本。

## 其他主要的大模型

除了 GPT 和 Claude，市面上还有很多大模型。大致可以分成两类：

**闭源模型**：模型本身不公开，只能通过厂商的 API 使用。

| 模型 | 厂商 | 说明 |
|---|---|---|
| GPT | OpenAI（美国） | 最早流行起来的大模型，ChatGPT 背后的模型 |
| Claude | Anthropic（美国） | 擅长写代码、长文本和智能体任务 |
| Gemini | Google（美国） | 和 Google 的搜索、文档等产品结合紧密 |

**开放权重模型**：模型的参数公开发布，可以下载下来，在自己的电脑或服务器上运行。很多厂商同时也提供 API。

| 模型 | 厂商 | 说明 |
|---|---|---|
| Llama | Meta（美国） | 影响很大的开放模型系列 |
| Mistral | Mistral AI（法国） | 欧洲的代表性模型 |
| DeepSeek | 深度求索（中国） | 以低成本、高性能著称 |
| Qwen（通义千问） | 阿里巴巴（中国） | 型号齐全，从小模型到大模型都有 |

国内还有 Kimi（月之暗面）、GLM（智谱）、豆包（字节跳动）等。大模型更新得很快，各家的具体型号几个月就会变一次，用的时候以各家官网为准。

有两点对我们写代码特别有用：

- **很多厂商的 API 都兼容 OpenAI 的格式。** 就像第 1 章提到的，用 `OpenAI` 客户端，把 `base_url` 换成它们的地址、`api_key` 换成它们的 Key、`model` 换成它们的模型名，就能直接调用。也就是说，`OpenAIProvider` 稍作修改，就能用上这些模型。
- **开放权重模型可以在本地运行。** 比如用 [Ollama](https://ollama.com) 这样的工具，在自己电脑上下载并运行模型，它也提供兼容 OpenAI 格式的接口。不用花钱，数据也不会离开你的电脑，只是本地的小模型在能力上通常比不过大厂的旗舰模型。

## 动手试试

- 给 `call` 传入 `model` 参数，换一个模型试试，比较一下回答有什么不同。
- 如果你有其他兼容 OpenAI 格式的模型（比如国产模型的 API，或者本地的 Ollama），模仿 `OpenAIProvider` 写一个新的子类：只要改 `base_url`、`api_key` 和默认模型名。

## 小结

- 不同厂商的 API 大同小异：都有 `model` 和 `messages`，但调用方法、必填参数和回答的位置各不相同。
- 我们用**抽象类** `LLMProvider` 规定统一的接口 `call`，再为每家厂商写一个子类，把各自的细节藏在子类里面。
- 学习助手的其他代码只依赖 `LLMProvider`，换模型只需要换一个 `provider`。
- 大模型分为闭源模型和开放权重模型；很多模型兼容 OpenAI 的 API 格式，也可以在本地运行。

现在的 `call` 只能传入一句话。可真正的对话，有系统的设定、有用户的提问、有模型的回答。下一章，我们来设计**消息**：用一组类来表示对话中各种角色说的话。